# 02 — Bronze Layer Ingestion (Auto Loader & Delta Lake): Real Estate (Property Listings, Market Valuation & AVM Pricing)

**Purpose:** Incrementally ingest raw JSON (`property_listings`) and CSV (`property_sales_history`) files from the Unity Catalog Volume into `real_estate_prod.bronze` Delta tables using Databricks Auto Loader (`cloudFiles`) with `availableNow=True` trigger and ingestion audit metadata.

In [ ]:
# Databricks Widget Parameters (overridden by Lakeflow Jobs / Databricks Asset Bundles)
dbutils.widgets.text("catalog_name", "real_estate_prod", "Unity Catalog Name")
CATALOG = dbutils.widgets.get("catalog_name")
VOLUME_PATH = f"/Volumes/{CATALOG}/bronze/raw_landing"
CHECKPOINT_PATH = f"/Volumes/{CATALOG}/bronze/checkpoints"
print(f"Active Catalog: {CATALOG} | Landing Volume: {VOLUME_PATH}")

## Step 1: Ingest `property_listings` (JSON) into `real_estate_prod.bronze.bronze_property_listings`

In [ ]:
from pyspark.sql import functions as F

raw_1_path = f"{VOLUME_PATH}/property_listings"
chk_1_path = f"{CHECKPOINT_PATH}/bronze_property_listings"
schema_1_path = f"{CHECKPOINT_PATH}/schemas/bronze_property_listings"

df_stream_1 = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", schema_1_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(raw_1_path)
    .withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

(
    df_stream_1.writeStream
    .format("delta")
    .option("checkpointLocation", chk_1_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.bronze.bronze_property_listings")
    .awaitTermination()
)
print(f"Bronze table {CATALOG}.bronze.bronze_property_listings row count:", spark.table(f"{CATALOG}.bronze.bronze_property_listings").count())

## Step 2: Ingest `property_sales_history` (CSV) into `real_estate_prod.bronze.bronze_property_sales`

In [ ]:
raw_2_path = f"{VOLUME_PATH}/property_sales_history"
chk_2_path = f"{CHECKPOINT_PATH}/bronze_property_sales"
schema_2_path = f"{CHECKPOINT_PATH}/schemas/bronze_property_sales"

df_stream_2 = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", schema_2_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(raw_2_path)
    .withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

(
    df_stream_2.writeStream
    .format("delta")
    .option("checkpointLocation", chk_2_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.bronze.bronze_property_sales")
    .awaitTermination()
)
print(f"Bronze table {CATALOG}.bronze.bronze_property_sales row count:", spark.table(f"{CATALOG}.bronze.bronze_property_sales").count())

## Step 3: Inspect Bronze Tables & Audit Columns

In [ ]:
display(spark.table(f"{CATALOG}.bronze.bronze_property_listings").limit(5))
display(spark.table(f"{CATALOG}.bronze.bronze_property_sales").limit(5))